# H5 Rounds (orchestrator-side re-reading, for now)

Frozen before the run.

**Claim.** Re-reading the saved worker caches with the partial findings repairs split failures, and the rounds needed ≈ crossings.

**Arms** (thinking off; split placements; up to 300 source questions per cell, under each seed in `SEEDS` (3 seeds = 3x the cost of seed 0 alone)):
- K0.
- scan with 1, 2 or 3 rounds: the orchestrator reads [S | one worker block] at a time, carries one-line notes forward, and answers on the last block.
- scan with an uncapped scratchpad, 1 round.

**Prediction.**
- On the toy, 1 round succeeds when the chain is read in order (AB) and fails reversed (BA), and 2 rounds repairs BA.
- Rounds needed ≈ crossings: at 2 crossings, 2 rounds ≫ 1 round.
- On MuSiQue the gain from rounds is small, because the final composition step is the bottleneck. Answer-in-notes ≫ answered-correctly.

**Decision rule.** Rounds ≈ crossings holds if accuracy at R rounds plateaus at R = crossings on the toy (paired CI for R vs R−1 excludes 0 up to R = crossings, then doesn't).

**Pilot (Qwen2.5-7B):**
- Toy, 2 crossings: 1 pass in order 3, reversed 14; 2 passes reversed 62; old scratchpad reversed 100/96.
- MuSiQue 2-step: reversed 1 pass 34, K passes 39.
- MuSiQue 3-step: answer present in the notes 120/200, answered correctly 29.

**Not built yet:**
- Worker-side rounds: broadcast the finding, and each worker re-selects from its saved cache. These need a compressed channel (K4).
- The oracle-query arm.
- The matched-budget control.

In [ ]:
# Settings. Headless full run: see README "Running".
MODEL = "Qwen/Qwen3-4B"   # any repo in latent_dispatch.model.MODELS (the model matrix: README "Models")
DTYPE = "bfloat16"
SMOKE = True        # True: 2 questions per cell, to read raw outputs first. False: the full design.
SEEDS = [0, 1, 2]   # data seeds: each source question in 3 distractor layouts (SMOKE uses the first only)
N_PER_CELL = None   # questions per (dataset, qtype, placement, seed) cell; None = this notebook's full design
SKIP = []           # arm names to leave out this time, e.g. ["t1_1024"] for a quick pass

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from latent_dispatch import analysis, arms, data
from latent_dispatch.model import Model
from latent_dispatch.results import load_results, results_path, run
from latent_dispatch.score import contains_answer, normalize

## Questions and arms

In [ ]:
SPLIT = ["AB", "BA", "AAB", "ABA", "AABB", "ABBA", "ABAB"]     # rounds have nothing to repair at 0 crossings
QTYPES = ["hop1", "hop2", "2hop", "3hop1", "4hop1"]           # toy hop0 is 0 crossings at every placement
seeds = SEEDS[:1] if SMOKE else SEEDS
qs = [q for name in ("toy", "toy_d0", "musique") for q in data.load(name, seed=seeds, placement=SPLIT, qtype=QTYPES)]
qs = data.per_cell(qs, 2 if SMOKE else (N_PER_CELL or 300))
print(len(qs), "questions")

ARMS = {
    "k0": (arms.k0, {}),
    "scan_1": (arms.scan, {"rounds": 1}),
    "scan_2": (arms.scan, {"rounds": 2}),
    "scan_3": (arms.scan, {"rounds": 3}),
    "scan_scratch": (arms.scan, {"rounds": 1, "note_style": "scratch", "note_tokens": 1024, "note_chars": 0}),
}

In [ ]:
model = Model.load(MODEL, dtype=DTYPE)
for name, (arm, options) in ARMS.items():
    if name not in SKIP:
        run(model, qs, name, arm, **options)

In [ ]:
df = load_results(qs, ARMS, MODEL, DTYPE)
display(analysis.audit(df))     # usable outputs? boxed answers and token-cap cuts, per arm
if SMOKE:
    analysis.peek(df)

## Accuracy

In [ ]:
analysis.accuracy(df, by=("dataset", "qtype", "placement"))

## Does each extra round help?

In [ ]:
for a, b in (("scan_2", "scan_1"), ("scan_3", "scan_2")):
    print(f"{a} − {b}")
    display(analysis.paired(df, a, b, by=("dataset", "qtype", "placement")))

In [ ]:
acc = (df[df.arm.str.match(r"scan_\d")].assign(rounds=lambda x: x.arm.str[-1].astype(int))
       .groupby(["dataset", "crossings", "rounds"]).em.mean().mul(100).reset_index())
fig, axes = plt.subplots(1, acc.dataset.nunique(), figsize=(4 * acc.dataset.nunique(), 3.5), sharey=True, squeeze=False)
for ax, (ds, g) in zip(axes[0], acc.groupby("dataset")):
    for c, gc in g.groupby("crossings"):
        ax.plot(gc.rounds, gc.em, marker="o", label=f"{c} crossings")
    ax.set(title=ds, xlabel="rounds", xticks=[1, 2, 3])
axes[0][0].set_ylabel("EM")
axes[0][0].legend()
plt.show()

## Answer present in the notes vs answered correctly

In [ ]:
rows = []
for (arm, ds), g in df[df.arm.str.startswith("scan")].groupby(["arm", "dataset"]):
    in_notes = [contains_answer(" ".join(n), a) for n, a in zip(g.notes, g.answers)]
    rows.append({"arm": arm, "dataset": ds, "answer in notes": round(100 * sum(in_notes) / len(g), 1),
                 "answered correctly": round(100 * g.em.mean(), 1)})
pd.DataFrame(rows)

## Findings

_(fill in after the full run)_